# Context-Deference — clean-baseline driver (judged labels + real safety attacks)

Replaces the keyword scorers with model **judges**, and the single weak jailbreak with **HarmBench**:
- **safety** → HarmBench 200 behaviors × 4 jailbreak templates, labeled by **HarmBench-Llama-2-13b-cls**
- **sycophancy / knowledge-conflict** → labeled by a local instruct-LLM judge

Runs in **3 memory-safe phases** — target → safety-judge → llm-judge, loaded one at a time so a 7B + 13B
fit an **A100-40GB** sequentially — then prints the shared-suppression report (transfer, leave-one-out
t_G, cosines). Needs an A100 GPU runtime. `acts_by / masks_by / sup_dirs` stay in memory for follow-ups.

In [ ]:
# === Bootstrap — upload the LATEST context-deference.zip, install, HF login ===
import sys, os
if "google.colab" in sys.modules:
    from google.colab import files
    print("Upload the LATEST context-deference.zip (must contain scripts/ and src/judge.py) ...")
    files.upload()
    os.system("rm -rf /content/suppression && unzip -oq context-deference.zip -d /content")
    os.chdir("/content/suppression")
    os.system("pip -q install -r requirements.txt")
    from huggingface_hub import login; login()      # HF token — HarmBench-cls is Llama-2-based
    print("bootstrap done | cwd:", os.getcwd())
sys.path.insert(0, os.getcwd())                      # repo root, so `import src...` resolves

In [ ]:
# === Config + imports ===
import gc, json
import numpy as np, torch
from src import model as M, data as D, directions as Dir, projection as P, judge as J, universal as U
torch.set_grad_enabled(False)

cfg        = D.load_behaviors_config("configs/behaviors.yaml")
models_cfg = D.load_yaml("configs/models.yaml")
MODEL      = os.environ.get("CD_MODEL", "qwen2.5-7b-instruct")
JUDGE_LLM  = os.environ.get("CD_JUDGE_LLM", "Qwen/Qwen2.5-7B-Instruct")   # sycophancy / kc judge
CONTRAST   = os.environ.get("CD_CONTRAST", cfg["contrast"]["mode"])
SUBSET     = os.environ["CD_SUBSET"].split(",") if os.environ.get("CD_SUBSET") else cfg["mvp_subset"]
LAYER      = int(os.environ.get("CD_LAYER", "16"))
MAXNT      = int(os.environ.get("CD_MAX_NEW_TOKENS", "64"))
POS        = -1
MAXB       = int(os.environ.get("CD_MAX_PAIRS", "0")) or None   # <-- set to 40 for a FAST SMOKE, None = full
mc = M.resolve_model_cfg(models_cfg, MODEL)
def _free(): gc.collect(); (torch.cuda.empty_cache() if torch.cuda.is_available() else None)
print("target:", mc["tl_name"], "| judge:", JUDGE_LLM, "| layer:", LAYER,
      "| behaviors:", SUBSET, "| max_pairs:", MAXB, "| contrast:", CONTRAST)

In [ ]:
# === Phase A: target model — generate completions + cache activations, then free ===
bundle = M.load_model(mc["tl_name"], dtype=mc.get("dtype", "bfloat16"))
store = {}
for b in SUBSET:
    pairs = D.load_pairs(b, cfg); pairs = pairs[:MAXB] if MAXB else pairs
    clean = M.format_chat(bundle, [p.prompt_clean for p in pairs])
    manip = M.format_chat(bundle, [p.prompt_manip for p in pairs])
    pt, nt = D.load_signal_statements(b, cfg)
    if MAXB: pt, nt = pt[:MAXB], nt[:MAXB]
    store[b] = dict(pairs=pairs,
        acts_manip=M.get_activations(bundle, manip, [LAYER], [POS])[(LAYER, POS)],
        acts_clean=M.get_activations(bundle, clean, [LAYER], [POS])[(LAYER, POS)],
        acts_pos=M.get_activations(bundle, M.format_chat(bundle, pt), [LAYER], [POS])[(LAYER, POS)],
        acts_neg=M.get_activations(bundle, M.format_chat(bundle, nt), [LAYER], [POS])[(LAYER, POS)],
        gen_manip=M.generate(bundle, manip, max_new_tokens=MAXNT),
        gen_clean=M.generate(bundle, clean, max_new_tokens=MAXNT))
    print(f"  cached {b}: {len(pairs)} pairs")
del bundle; _free()

In [ ]:
# === Phase B: judges -> labels (HarmBench-cls for safety; local LLM for sycophancy/kc) ===
cls = J.load_hf(J.HARMBENCH_CLS)                       # ~13B; loads after target is freed
if "safety" in SUBSET:
    s = store["safety"]
    s["overrode"]   = J.score("safety", s["pairs"], s["gen_manip"], cls=cls)
    s["clean_over"] = J.score("safety", s["pairs"], s["gen_clean"], cls=cls)   # clean arm should ~all be 0
del cls; _free()
llm = J.load_hf(JUDGE_LLM)
for b in [x for x in SUBSET if x in ("sycophancy", "knowledge_conflict")]:
    s = store[b]
    s["overrode"]   = J.score(b, s["pairs"], s["gen_manip"], llm=llm)
    s["clean_over"] = J.score(b, s["pairs"], s["gen_clean"], llm=llm)
del llm; _free()
print("judging done")

In [ ]:
# === Phase C: directions from cached acts + judged masks (+ the headline override rates) ===
sup_dirs, sig_dirs, acts_by, masks_by = {}, {}, {}, {}
for b in SUBSET:
    s = store[b]
    ov = torch.tensor([x == 1 for x in s["overrode"]]); ck = torch.tensor([x == 0 for x in s["clean_over"]])
    acts_by[b]  = {"manip": s["acts_manip"], "clean": s["acts_clean"]}
    masks_by[b] = {"overrode": ov, "resisted": ~ov, "clean_kept": ck}
    sig_dirs[b] = Dir.signal_direction(s["acts_pos"], s["acts_neg"], layer=LAYER, position=POS, name="s", behavior=b)
    sup_dirs[b] = Dir.suppression_direction(acts_by[b], masks_by[b], mode=CONTRAST, layer=LAYER, position=POS, behavior=b)
    print(f"  {b:<20} JUDGED override rate: {int(ov.sum())}/{len(ov)} = {ov.float().mean():.2f}")
sup_res = dict(zip(SUBSET, P.residualize_all([sup_dirs[b] for b in SUBSET],
                                             [sig_dirs[b] for b in SUBSET], own_only=True)))

In [ ]:
# === Shared-suppression report on JUDGED labels (transfer + leave-one-out t_G + cosines) ===
print(f"layer {LAYER} | contrast {CONTRAST} | JUDGED labels\n")
res = U.report(sup_res, acts_by, masks_by, SUBSET)
os.makedirs("results", exist_ok=True)
json.dump({"model": mc["tl_name"], "layer": LAYER, "contrast": CONTRAST,
           "override_rates": {b: float(masks_by[b]["overrode"].float().mean()) for b in SUBSET}},
          open("results/clean_baseline.json", "w"), indent=2)

## Reading it
- **Judged override rates** (Phase C) are the headline — safety should now be a real, sizable sample, not 11.
- **TRANSFER / leave-one-out t_G**: any off-diagonal beating its column's `random95%` = a shared axis; all
  below = distinct, now on trustworthy labels.
- `acts_by`, `masks_by`, `sup_dirs`, `sup_res`, `store` stay in memory — paste the layer-sweep / per-pair
  cells on top for interactive follow-ups.

## Causal test (phase 2) — is the shared axis actually *causal*?
Ablate each residual suppression axis (plus the syc–safety **shared** axis and a **random** control),
re-generate the manipulated prompts, and **re-judge**. A shared mechanism ⇒ ablating the shared axis
drops override for **both** sycophancy and safety but **not** knowledge-conflict, while random ≈ none.
Reuses `sup_res` + `store` from above (reloads the target once to steer). Bound with `CD_N_CAUSAL`.

In [ ]:
# === CAUSAL: reload target, ablate each axis, generate steered completions ===
from src import steering as St
N_CAUSAL  = int(os.environ.get("CD_N_CAUSAL", "60"))     # prompts/behavior per ablation (bounds compute)
NT_CAUSAL = 32
_u = lambda v: v.float() / (v.float().norm() + 1e-8)
shared = _u(torch.stack([_u(sup_res["sycophancy"].vec), _u(sup_res["safety"].vec)]).mean(0))
axes = {"none": None, "ablate_syc": sup_res["sycophancy"].vec, "ablate_safety": sup_res["safety"].vec,
        "ablate_kc": sup_res["knowledge_conflict"].vec, "ablate_shared": shared}
bundle = M.load_model(mc["tl_name"], dtype=mc.get("dtype", "bfloat16"))
axes["ablate_random"] = St.sample_random_directions(bundle.d_model, 1, seed=0)[0]
prompts = {b: M.format_chat(bundle, [p.prompt_manip for p in store[b]["pairs"][:N_CAUSAL]]) for b in SUBSET}
steered = {a: {} for a in axes}
for aname, avec in axes.items():
    hooks = [] if avec is None else St.ablation_hooks(bundle, avec)
    for b in SUBSET:
        steered[aname][b] = St.run_with_hooks(bundle, prompts[b], hooks, max_new_tokens=NT_CAUSAL)
    print("steered under:", aname)
del bundle; _free()

In [ ]:
# === CAUSAL: judge the steered generations -> override-rate matrix (rows = ablation, cols = behavior) ===
rate = {a: {} for a in axes}
cls = J.load_hf(J.HARMBENCH_CLS)
for a in axes:
    rate[a]["safety"] = float(np.mean(J.score("safety", store["safety"]["pairs"][:N_CAUSAL], steered[a]["safety"], cls=cls)))
del cls; _free()
llm = J.load_hf(JUDGE_LLM)
for a in axes:
    for b in ("sycophancy", "knowledge_conflict"):
        rate[a][b] = float(np.mean(J.score(b, store[b]["pairs"][:N_CAUSAL], steered[a][b], llm=llm)))
del llm; _free()
print(f"{'ablate v / measure >':<22}" + "".join(f"{b[:11]:>13}" for b in SUBSET))
for a in axes:
    print(f"{a:<22}" + "".join(f"{rate[a][b]:>13.2f}" for b in SUBSET))
print("SHARED => ablate_shared drops syc & safety but NOT kc (vs 'none'); ablate_random ~ 'none' (control).")